# TB-AWFNO temporal-basis rank ablation — 2D Cahn–Hilliard

This notebook trains **only TB-AWFNO** for the two integer temporal-basis ranks `M=K//2` and `M=(2*K)//3`. Here **K=5**, hence **M=2, 3**. The original benchmark implementation is downloaded from GitHub; no other neural operators are trained.

**Default:** `N_train=400`, `epochs=400`, seed 1234, original validation/test split and physical absolute L² metrics. Both ranks use the same model architecture except for the coefficient-output dimension `M`. Each rank has a distinct Google Drive directory. All POD bases use **training trajectories only**, and test data are evaluated only after best-validation checkpoint selection.

**Execution status:** No GPU runs were performed when this notebook was created. Before running, mount Drive, confirm dataset location and GPU memory, and review the estimated computational expense. This is an ablation at one training sample size, *not* a full sample-size sweep.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, json, hashlib, urllib.request
from pathlib import Path
import torch, pandas as pd, matplotlib.pyplot as plt
assert torch.cuda.is_available(), 'Select GPU: Runtime -> Change runtime type'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Editable rank-ablation configuration
PDE="cahn_hilliard_2d"
K=5
N_TRAIN=400
EPOCHS=400
SEED=1234
DATA_PATH=Path("/content/drive/MyDrive/anus/cahn_hilliard_train_grid_32x32.pt")
OUTPUT_ROOT=Path("/content/drive/MyDrive/AWFNO_TB_rank_sweep/cahn_hilliard_2d")
RANKS=sorted(set((K//2, (2*K)//3)))  # integer floor, never fractions
FORCE_RETRAIN=False  # keep False to reuse completed checkpoints
assert all(isinstance(m,int) and 1 <= m <= K for m in RANKS)
assert DATA_PATH.is_file(), f'Missing dataset: {DATA_PATH}'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('K:',K,'M ranks:',RANKS,'N train:',N_TRAIN,'epochs:',EPOCHS)
print('Dataset:',DATA_PATH,'size GiB:',round(DATA_PATH.stat().st_size/2**30,3))
print('Output:',OUTPUT_ROOT)


In [ ]:
# Download the original experiment source (not a reimplementation).
SOURCE_URL='https://raw.githubusercontent.com/dimigourz/AWFNO/main/experiments/cahn_hilliard_2d/ablations_tb.py'
SOURCE_PATH=Path('/content/tb_reference_cahn_hilliard_2d.py')
request=urllib.request.Request(SOURCE_URL,headers={'Cache-Control':'no-cache'})
SOURCE_BYTES=urllib.request.urlopen(request,timeout=120).read()
assert len(SOURCE_BYTES)>20000, 'Incomplete experiment source'
SOURCE_PATH.write_bytes(SOURCE_BYTES)
import py_compile
py_compile.compile(str(SOURCE_PATH),doraise=True)
print('Verified benchmark Python source:',SOURCE_URL)
print('SHA256:',hashlib.sha256(SOURCE_BYTES).hexdigest())


In [ ]:
# Two separate ranks; safely skip completed results on a resumed Colab session.
DEFAULT_ENV={
  "REVIEWER_SUITE": "tb",
  "REVIEWER_MODELS": "TB_AWFNO_LFL",
  "REVIEWER_N_VAL": "30",
  "REVIEWER_N_TEST": "70",
  "REVIEWER_BATCH_SIZE": "32",
  "REVIEWER_LR": "0.001",
  "REVIEWER_WEIGHT_DECAY": "1e-6",
  "REVIEWER_FINAL_LOSS_WEIGHT": "1.0",
  "REVIEWER_REQUIRE_PM_CACHE": "0",
  "REVIEWER_SAVE_ROLLOUTS": "0",
  "REVIEWER_WINDOW_K": "5",
  "REVIEWER_SEEDS": "1234",
  "EXPERIMENT_NAME": "tb_rank",
  "SPLIT_SEED": "2026"
}
for M in RANKS:
    rank_dir=OUTPUT_ROOT/f'N{N_TRAIN}'/f'M{M}'
    rank_dir.mkdir(parents=True,exist_ok=True)
    marker=rank_dir/'result.json'
    if marker.exists() and not FORCE_RETRAIN:
        print('SKIPPING completed rank M=',M,'result:',marker,flush=True)
        continue
    env=os.environ.copy()
    env.update({k:str(v) for k,v in DEFAULT_ENV.items()})
    env.update({'DATA_PATH':str(DATA_PATH), 'DATA_FILE':str(DATA_PATH),
                'OUTPUT_ROOT':str(rank_dir), 'REVIEWER_OUTPUT_ROOT':str(rank_dir),
                'N_TIME_MODES':str(M), 'REVIEWER_TIME_MODES':str(M),
                'EPOCHS':str(EPOCHS), 'REVIEWER_EPOCHS':str(EPOCHS),
                'REVIEWER_N_TRAIN':str(N_TRAIN),'WINDOW_K':str(K),
                'TB_PDE':PDE,'TB_N_TRAIN':str(N_TRAIN), 'TB_WINDOW_K':str(K),
                'TB_RANK_M':str(M),'TB_SEED':str(SEED),
                'TB_RANK_OUTPUT':str(rank_dir), 'TB_SOURCE_FILE':str(SOURCE_PATH),
                'FORCE_RETRAIN':'1' if FORCE_RETRAIN else '0',
                'PYTHONUNBUFFERED':'1'})
    if PDE=='navier_stokes_2d':
        env.update({'NS_DATA_DIR':str(DATA_PATH.parent),'NS_OUTPUT_ROOT':str(rank_dir),
                    'NS_SIZES':str(N_TRAIN),'NS_TB_RANK':str(M),
                    'NS_K':str(K),'NS_STRIDE':str(K),'NS_EPOCHS':str(EPOCHS)})
    if PDE=='burgers_1d':
        env['RUN_SAMPLE_SIZES']=str(N_TRAIN)
    if PDE=='boussinesq_2d':
        env['RUN_SAMPLE_SIZES']=str(N_TRAIN)
    print('\\n==========',PDE,'K=',K,'M=',M,'==========',flush=True)
    command=[sys.executable,'-u',str(SOURCE_PATH)]
    log_path=rank_dir/'run.log'
    with log_path.open('a',encoding='utf-8') as log:
        with subprocess.Popen(command,env=env,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1,errors='replace') as proc:
            for line in proc.stdout:
                print(line,end='',flush=True)
                log.write(line);log.flush()
            status=proc.wait()
    if status:
        raise RuntimeError(f'Rank M={M} exited code {status}. Review {log_path}')
    raw=list(rank_dir.glob('tb_rank_N*/summary_tb_raw.csv'))
    assert len(raw)==1, f'Expected one TB summary CSV in {rank_dir}, found {raw}'
    frame=pd.read_csv(raw[0])
    assert len(frame)==1 and frame.iloc[0]['model']=='TB_AWFNO_LFL'
    row=frame.iloc[0]
    result={'pde':PDE,'model':'TB-AWFNO','K':K,'M':M,'N_train':N_TRAIN,'seed':SEED,
            'full_l2_mean':float(row['full_l2_mean']),
            'final_l2_mean':float(row['final_l2_mean']),
            'parameters':int(row['tensor_parameter_count']),
            'train_seconds':float(row['train_time_sec']),
            'source_sha256':hashlib.sha256(SOURCE_BYTES).hexdigest()}
    marker.write_text(json.dumps(result,indent=2))
    assert marker.exists(), f'No result file after rank M={M}'
    print('SAVED',marker,flush=True)


In [ ]:
records=[]
for M in RANKS:
    result_path=OUTPUT_ROOT/f'N{N_TRAIN}'/f'M{M}'/'result.json'
    if result_path.is_file():
        records.append(json.loads(result_path.read_text()))
if records:
    df=pd.DataFrame(records).sort_values('M')
    table=OUTPUT_ROOT/f'N{N_TRAIN}'/'tb_rank_comparison.csv'
    table.parent.mkdir(parents=True,exist_ok=True)
    df.to_csv(table,index=False)
    display(df[[x for x in ('pde','N_train','K','M','parameters','full_l2_mean','final_l2_mean','train_seconds') if x in df.columns]])
    print('Saved:',table)
    if len(df)==2:
        fig, ax=plt.subplots(figsize=(7,4))
        ax.plot(df['M'],df['full_l2_mean'],marker='o',label='Full trajectory L2')
        ax.plot(df['M'],df['final_l2_mean'],marker='s',label='Final-time L2')
        ax.set(xlabel='Temporal POD rank M',ylabel='Absolute physical L2',
               title=f'{PDE}: TB-AWFNO rank ablation, K={K}, N={N_TRAIN}')
        ax.set_xticks(df['M'])
        ax.legend();ax.grid(alpha=.25);fig.tight_layout()
        fig.savefig(table.with_suffix('.png'),dpi=160)
        plt.show()
else:
    print('No completed rank results yet.')
